# Train LipForensics tr?n dataset crop m?i

Notebook n?y hu?n luy?n ki?n tr?c LipForensics ch?nh th?c (CVPR 2021) tr?n dataset ?? ti?n x? l? c?a project. Dataset c?n c? `train/pytmp/{real,fake}/...`, c?c th? m?c chunk ch?a ?nh `.jpg`, v? c? th? c? `train_split.csv`, `valid_split.csv`, `test_imbalance_split.csv`.

**L?u ?:** LipForensics l? m? h?nh th? gi?c, kh?ng d?ng audio. Notebook gi? crop m?i c?a b?n nh?ng d?ng 25 frame grayscale 88x88 theo chu?n LipForensics.

In [ ]:
# Ch?n Runtime -> Change runtime type -> GPU tr??c khi ch?y
!nvidia-smi -L
!pip -q install gdown scikit-learn pandas opencv-python-headless tqdm

In [ ]:
import os, sys, glob, tarfile, shutil, random, json, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FILE_ID = '1-4l5XXe33aldyW_DQI6buwU2z2ozZx7f'
DRIVE_FILE = '/content/drive/MyDrive/lipforensics_dataset.tar'
DATA_DIR = '/content/lipforensics_data'
REPO_DIR = '/content/LipForensics'
OUT_DIR = '/content/drive/MyDrive/lipforensics_runs'
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(DRIVE_FILE):
    !gdown --id {DRIVE_FILE_ID} -O {DRIVE_FILE}
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ahaliassos/LipForensics.git {REPO_DIR}
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    with tarfile.open(DRIVE_FILE, 'r:*') as tar:
        tar.extractall(DATA_DIR)
print('Dataset archive:', DRIVE_FILE)
print('Dataset extracted:', DATA_DIR)
print('Top-level:', os.listdir(DATA_DIR)[:20])

In [ ]:
# T?m th? m?c ch?a train/pytmp v? c?c split CSV
def find_root(base):
    candidates = [Path(base)] + [p for p in Path(base).rglob('*') if p.is_dir()]
    for p in candidates:
        if (p/'train'/'pytmp').is_dir():
            return p
    raise FileNotFoundError('Kh?ng t?m th?y train/pytmp trong dataset ?? gi?i n?n.')
ROOT = find_root(DATA_DIR)
print('DATA_ROOT =', ROOT)
for p in sorted(ROOT.rglob('*.csv')):
    print('CSV:', p.relative_to(ROOT))
for label in ('real','fake'):
    p = ROOT/'train'/'pytmp'/label
    print(label, 'videos:', len([x for x in p.iterdir() if x.is_dir()]) if p.exists() else 0)

In [ ]:
# N?u tar kh?ng c? split CSV, t?o split theo video ?? tr?nh leakage gi?a c?c chunk
import csv
random.seed(42)
CSV_NAMES = {'train':'train_split.csv', 'valid':'valid_split.csv', 'test':'test_imbalance_split.csv'}
def rows_for(videos, label):
    rows=[]
    base = ROOT/'train'/'pytmp'/label
    for video in videos:
        vd = base/video
        for chunk in sorted([x for x in vd.iterdir() if x.is_dir()]):
            audio = vd/(chunk.name + '.wav')
            if audio.exists():
                rows.append([str(chunk.relative_to(ROOT)), str(audio.relative_to(ROOT)), label])
            else:
                # LipForensics ch? c?n ?nh; ch?p nh?n chunk kh?ng c? audio
                imgs = list(chunk.glob('*.jpg')) + list(chunk.glob('*.png'))
                if imgs: rows.append([str(chunk.relative_to(ROOT)), '', label])
    return rows
for split, fname in CSV_NAMES.items():
    target = ROOT/fname
    if target.exists(): continue
    all_rows=[]
    for label in ('real','fake'):
        videos = [x.name for x in (ROOT/'train'/'pytmp'/label).iterdir() if x.is_dir()]
        random.shuffle(videos)
        n=len(videos); a=int(.70*n); b=int(.85*n)
        selected = videos if split=='train' else videos[a:b] if split=='valid' else videos[b:]
        all_rows += rows_for(selected, label)
    random.shuffle(all_rows)
    with open(target,'w',newline='') as f: csv.writer(f).writerows(all_rows)
    print('Created', target, len(all_rows))
print('Splits ready')

In [ ]:
# Import official LipForensics code
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from tqdm.auto import tqdm
from models.spatiotemporal_net import Lipreading

# Official model config
with open(REPO_DIR + '/models/configs/lrw_resnet18_mstcn.json') as f: mc = json.load(f)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FRAMES = 25
SIZE = 88
print('Device:', DEVICE, 'frames:', FRAMES, 'size:', SIZE)

In [ ]:
class MouthClipDataset(Dataset):
    def __init__(self, csv_path, train=False):
        self.train=train; self.items=[]
        with open(csv_path, newline='') as f:
            for row in csv.reader(f):
                if len(row) >= 3: self.items.append((row[0], int(row[2].lower() in ('fake','1','true'))))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        rel, label = self.items[i]
        folder = Path(rel) if Path(rel).is_absolute() else ROOT/rel
        files = sorted(list(folder.glob('*.jpg')) + list(folder.glob('*.png')))
        if len(files) < FRAMES: raise RuntimeError(f'Chunk c? {len(files)} frame, c?n ?t nh?t {FRAMES}: {folder}')
        # L?y 25 frame ??u trong 30 frame ?? crop
        idx = np.linspace(0, len(files)-1, FRAMES).round().astype(int)
        frames=[]
        for j in idx:
            img=cv2.imread(str(files[j]), cv2.IMREAD_GRAYSCALE)
            if img is None: raise RuntimeError(f'Kh?ng ??c ???c ?nh: {files[j]}')
            img=cv2.resize(img,(SIZE,SIZE),interpolation=cv2.INTER_AREA)
            frames.append(torch.from_numpy(img).float().div(255).sub(.421).div(.165))
        x=torch.stack(frames,0).unsqueeze(0)
        if self.train and random.random()<0.5: x=torch.flip(x, dims=[3])
        return x, torch.tensor(label,dtype=torch.float32), str(folder)

train_ds=MouthClipDataset(str(ROOT/'train_split.csv'), True)
valid_ds=MouthClipDataset(str(ROOT/'valid_split.csv'))
test_ds=MouthClipDataset(str(ROOT/'test_imbalance_split.csv'))
print('Samples:', len(train_ds), len(valid_ds), len(test_ds))
train_loader=DataLoader(train_ds,batch_size=16,shuffle=True,num_workers=2,pin_memory=True)
valid_loader=DataLoader(valid_ds,batch_size=16,shuffle=False,num_workers=2,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=16,shuffle=False,num_workers=2,pin_memory=True)

In [ ]:
def make_model():
    tcn_options={'num_layers':mc['tcn_num_layers'],'kernel_size':mc['tcn_kernel_size'],'dropout':mc['tcn_dropout'],'dwpw':mc['tcn_dwpw'],'width_mult':mc['tcn_width_mult']}
    return Lipreading(num_classes=1,tcn_options=tcn_options,relu_type=mc['relu_type']).to(DEVICE)
model=make_model()
optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
criterion=nn.BCEWithLogitsLoss()
EPOCHS=20
BEST_PATH=OUT_DIR+'/lipforensics_best.pth'
print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
def run_epoch(loader, training):
    model.train(training); losses=[]; ys=[]; ps=[]
    for x,y,_ in tqdm(loader, leave=False):
        x,y=x.to(DEVICE,non_blocking=True),y.to(DEVICE,non_blocking=True)
        with torch.set_grad_enabled(training):
            logits=model(x,lengths=[FRAMES]*x.size(0)).squeeze(1)
            loss=criterion(logits,y)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); optimizer.step()
        losses.append(loss.item()*len(y)); ys.extend(y.detach().cpu().numpy()); ps.extend(torch.sigmoid(logits).detach().cpu().numpy())
    auc=roc_auc_score(ys,ps) if len(set(ys))==2 else float('nan')
    return sum(losses)/len(loader.dataset), auc, accuracy_score(ys,np.array(ps)>=.5), f1_score(ys,np.array(ps)>=.5,zero_division=0)

best=-1; history=[]
for epoch in range(1,EPOCHS+1):
    tr=run_epoch(train_loader,True); va=run_epoch(valid_loader,False)
    history.append((epoch,*tr,*va)); print(f'Epoch {epoch:02d} train loss/auc/acc/f1={tr} valid={va}')
    if va[1] > best:
        best=va[1]; torch.save({'model':model.state_dict(),'epoch':epoch,'valid_auc':best,'config':mc},BEST_PATH); print('Saved',BEST_PATH)

In [ ]:
# ??nh gi? checkpoint t?t nh?t tr?n test
ckpt=torch.load(BEST_PATH,map_location=DEVICE)
model.load_state_dict(ckpt['model'])
test=run_epoch(test_loader,False)
print('Best epoch:',ckpt['epoch'],'valid AUC:',ckpt['valid_auc'])
print('TEST loss, AUC, accuracy, F1:',test)
with open(OUT_DIR+'/history.json','w') as f: json.dump(history,f)

In [ ]:
# T?i checkpoint v? m?y
from google.colab import files
files.download(BEST_PATH)